# Template Eksperimen MSML — Preprocessing & EDA

Notebook ini mengikuti format template yang Anda berikan. Ikuti sel-sel berikut secara berurutan: impor library, muat dataset, inspeksi cepat, penanganan missing, encoding, scaling, feature engineering, pipeline, split & save, unit tests, dan konversi.

## 1. Perkenalan Dataset
Sumber dataset: CSV di folder `dataset/`. Notebook ini menunjukkan cara memuat CSV, inspeksi cepat, pra-pemrosesan, dan menyimpan hasil preprocessing.

In [ ]:
# 2. Import Library
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
import joblib

sns.set(style='whitegrid')
print('libraries imported')

## 3. Memuat Dataset
Baca file CSV (atau Parquet). Contoh file CSV: `dataset/ispu_dki_all.csv` dan varian lainnya.

In [ ]:
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset_cuaca'
print('dataset dir:', DATASET_DIR)
CSV_FILES = ['ispu_dki_all.csv','ispu_dki1.csv','ispu_dki2.csv','ispu_dki3.csv','ispu_dki4.csv','ispu_dki5.csv']

def read_datasets(csv_files=CSV_FILES):
    frames = []
    for f in csv_files:
        p = DATASET_DIR / f
        if not p.exists():
            print(f'warning: {p} missing')
            continue
        df = pd.read_csv(p)
        df.columns = [c.strip().lower() for c in df.columns]
        df['source_file'] = f
        frames.append(df)
    if not frames:
        raise FileNotFoundError('no dataset files found')
    return pd.concat(frames, ignore_index=True)

# raw_df = read_datasets()
# raw_df.head()

## 4. Inspeksi cepat data
Tampilkan `head()`, `info()`, `describe()` dan visualisasi missing values.

In [ ]:
def quick_inspect(df, n=5):
    display(df.head(n))
    print('info:')
    print(df.info())
    print('describe:')
    display(df.describe(include='all'))
    miss = df.isna().sum().sort_values(ascending=False)
    display(miss[miss>0])
    plt.figure(figsize=(6,4))
    sns.heatmap(df.isna(), cbar=False)
    plt.title('Missing value map')
    plt.show()

# Example usage after loading:
# df = read_datasets()
# quick_inspect(df)

## 5. Penanganan missing values
Contoh: `dropna`, `SimpleImputer` (mean/median/most_frequent) dan contoh imputasi berbasis model (sketch).

In [ ]:
def impute_example(df, numeric_cols):
    imputer = SimpleImputer(strategy='median')
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    return df, imputer

# usage: df, imp = impute_example(df, ['pm25','pm10'])

## 6. Encoding fitur kategorikal
Contoh OneHotEncoder, OrdinalEncoder, dan integrasi ke `ColumnTransformer`.

In [ ]:
# Example column transformer
categorical_cols = ['stasiun']
numeric_cols = ['pm25','pm10','so2','co','o3','no2']
num_t = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_t = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', num_t, numeric_cols), ('cat', cat_t, categorical_cols)])
print('column transformer ready')

## 7. Scaling / Normalization
Contoh `StandardScaler`, `MinMaxScaler`, `RobustScaler` pada fitur numerik dan visualisasi efeknya.

In [ ]:
from sklearn.preprocessing import RobustScaler
scalers = {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}
# Example: fit on numeric columns and plot distributions before/after

## 8. Feature engineering dan seleksi
Buat fitur turunan (tanggal→year/month/day) dan gunakan `SelectKBest` atau feature importance untuk seleksi.

In [ ]:
def add_date_features(df, date_col='tanggal'):
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['day'] = df[date_col].dt.day
    df['dayofweek'] = df[date_col].dt.dayofweek
    return df

# selection example:
# selector = SelectKBest(f_classif, k=10)

## 9. Split train/test dan pipeline dengan scikit-learn
Buat `Pipeline` lengkap dan simpan dengan `joblib`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])

# Example run:
# X = df[numeric_cols + categorical_cols]
# y = df['categori']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# model_pipe.fit(X_train, y_train)
# joblib.dump(model_pipe, 'Membangun_model/namadataset_preprocessing/pipeline.joblib')

## 10. Unit tests untuk fungsi preprocessing
Buat `tests/test_preprocessing.py` dengan contoh sederhana yang memanggil fungsi-fungsi preprocessing dan memeriksa bentuk output. (Di notebook ini berikan contoh sederhana).

In [ ]:
# Example pytest-like assertions (not running pytest here)
def _test_clean_example():
    df = pd.DataFrame({'tanggal':['2020-01-01','2020-01-02'], 'stasiun':['A','B'], 'pm25':[10,20], 'pm10':[20,30], 'so2':[1,2], 'co':[0.1,0.2], 'o3':[0.5,0.6], 'no2':[5,6], 'max':[10,20], 'critical':[0,1], 'categori':['LOW','MEDIUM']})
    df2 = add_date_features(df, 'tanggal')
    assert 'year' in df2.columns
    print('unit test sketch passed')

_test_clean_example()